# Publish the clean-split DistilBERT to the Hugging Face Hub

Runs on a Colab **CPU** runtime (no GPU, no training), in about 15–30 minutes, mostly waiting for
the demo to rebuild. It runs `scripts/publish_model.py`, which stops at the first failed check.

1. **Check, no writes.**
   - The run folder in Drive is the scored run: its records match the committed bundle.
   - An agreement check on 1,024 examples: the weights reproduce the saved test logits
     (absolute tolerance 1e-3, every prediction equal).
   - The Hub still holds the legacy model.
   - The demo answers, its Gradio version is read, and its patch applies.
2. **Upload.**
   - The legacy commit of `krimits/distilbert-hotel-reviews` gets the tag `legacy-split-v1`.
   - One commit holds the new model and its card. It is checked on the commit it returns:
     files, card, and the SHA-256 of the weights downloaded from it.
3. **Pin the demo.** `krimits/hotel-review-demo` loads the model at that exact commit, and its texts
   say which model it is.
4. **Check the demo.** It must report that commit and those weights, then label one positive and
   one negative review.

**Interrupted?** Run the notebook again from the top. `publish_state.json` in Drive records every
completed write. Each step checks the Hub again, and is skipped only if it was done exactly as
expected.

**Token.** It needs a Hugging Face token with write access: a Colab secret named `HF_TOKEN`, or
paste one when asked.

In [ ]:
RUN_DIR = "/content/drive/MyDrive/hotel-review-nlp-phase2/distilbert_v2"  # the random split, full fine-tune
STATE_PATH = "/content/drive/MyDrive/hotel-review-nlp-phase2/publish_state.json"
DRY_RUN = False  # True: only the checks and the card; nothing is written to the Hub or the demo

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")
assert Path(RUN_DIR, "model.safetensors").is_file(), f"no weights in {RUN_DIR}"

In [ ]:
import os
import shlex
import subprocess
import sys

REPO = Path("/content/hotel-review-nlp")
if not REPO.exists():
    subprocess.run(["git", "clone", "--quiet", "--branch", "claude/funny-rubin-31xgtb",
                    "https://github.com/krimits/hotel-review-nlp.git", str(REPO)], check=True)
os.chdir(REPO)
# The combination the tests ran with.
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "huggingface_hub==0.36.2",
                "transformers==4.56.2", "gradio_client==1.13.3"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--no-deps", "-e", "."], check=True)
print("publishing code:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())

In [ ]:
try:
    from google.colab import userdata

    token = userdata.get("HF_TOKEN")
except Exception:  # no secret, or no access to it
    token = None
if token:
    os.environ["HF_TOKEN"] = token
else:
    from huggingface_hub import notebook_login

    notebook_login()

In [ ]:
from huggingface_hub import HfApi

print("Hugging Face account:", HfApi().whoami()["name"])

## The test set, for the agreement check

In [ ]:
def run(command: str) -> None:
    """Run a command and show its output as it comes."""
    print("$", command, flush=True)
    with subprocess.Popen(command, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
    if process.returncode:
        raise RuntimeError(f"stopped with exit code {process.returncode}: {command}")


run("python scripts/fetch_booking_515k.py")
if not Path("data/processed_v2/test.parquet").exists():
    run("python -m reviewnlp.data.preprocess --config configs/baselines.yaml")

## 1. Check (no writes)

In [ ]:
PUBLISH = f"python -u scripts/publish_model.py {{step}} --run-dir {shlex.quote(RUN_DIR)} --state {shlex.quote(STATE_PATH)}"
run(PUBLISH.format(step="check") + " --card-out /content/model_card.md")
print(Path("/content/model_card.md").read_text(encoding="utf-8"))

## 2–4. Upload, pin the demo, check the demo

In [ ]:
if DRY_RUN:
    print("Dry run: nothing was written to the Hub or the demo.")
else:
    run(PUBLISH.format(step="upload"))
    run(PUBLISH.format(step="pin-demo"))
    run(PUBLISH.format(step="check-demo"))

In [ ]:
import json

state = json.loads(Path(STATE_PATH).read_text(encoding="utf-8"))
keys = ("code_commit", "old_hub_commit", "old_model_sha256", "model_commit", "new_model_sha256",
        "legacy_tag", "space_parent_commit", "space_commit", "gradio_version", "logits_check", "demo_check")
print(json.dumps({key: state.get(key) for key in keys}, indent=1))